## 1. get model and embedder ready

In [1]:
!wget -O src/embeddings/download.py https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/02-vector-search/embed/download.py

--2026-08-12 16:31:33--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8001::154, 2606:50c0:8002::154, 2606:50c0:8003::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8001::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘src/embeddings/download.py’

src/embeddings/down 100%[===================>]   1.34K  --.-KB/s    in 0.05s   

2026-08-12 16:31:34 (25.7 KB/s) - ‘src/embeddings/download.py’ saved [1376/1376]



In [2]:
!uv run python src/embeddings/download.py

  exists models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  exists models/Xenova/all-MiniLM-L6-v2/model.onnx


In [3]:
!wget -O src/embeddings/embedder.py https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/02-vector-search/embed/embedder.py

--2026-08-12 16:31:55--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8001::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1520 (1.5K) [text/plain]
Saving to: ‘src/embeddings/embedder.py’

src/embeddings/embe 100%[===================>]   1.48K  --.-KB/s    in 0.02s   

2026-08-12 16:31:56 (64.6 KB/s) - ‘src/embeddings/embedder.py’ saved [1520/1520]



In [7]:
from pathlib import Path

print("cwd:", Path.cwd())

cwd: /Users/chs/Documents/LLM-zoomcamp/Capstone/LyricLens


In [8]:
path = Path("models/Xenova/all-MiniLM-L6-v2")
print("resolved:", path.resolve())

resolved: /Users/chs/Documents/LLM-zoomcamp/Capstone/LyricLens/models/Xenova/all-MiniLM-L6-v2


In [9]:
print("directory exists:", path.exists())

directory exists: True


In [10]:
print("tokenizer exists:", (path / "tokenizer.json").exists())

tokenizer exists: True


In [14]:
from src.embeddings.embedder import Embedder

embedder = Embedder("models/Xenova/all-MiniLM-L6-v2")

---

## 2. in memory Vector Search

In [1]:
# autoreload every imported modules if their source files have changed.
%load_ext autoreload
%autoreload 2

In [2]:
from src.ingest_lyrics import load_lyrics

lyrics_df = load_lyrics()
lyrics_df

,title,performer,chart_weeks,wks_on_chart,peak_pos,plain_lyrics
0,$ex Appeal,Baby Keem Featuring Too $hort,[2026-03-07],1,69,"(Play with me, play with me)\n(Play with me, p..."
1,'98 Braves,Morgan Wallen,"[2023-03-18, 2023-03-25, 2023-04-01, 2023-04-0...",7,27,I remember sittin' at that house\nLivin' room ...
2,'Til You Can't,Cody Johnson,"[2021-10-23, 2021-10-30, 2021-11-06, 2022-01-0...",29,18,You can tell your old man\nYou'll do some larg...
3,'Tis The Damn Season,Taylor Swift,"[2020-12-26, 2021-01-02]",2,39,If I wanted to know who you were hanging with\...
4,(There's No Place Like) Home For The Holidays ...,Perry Como With Mitchell Ayers And His Orchestra,[2024-01-06],1,50,"Oh, there's no place like home for the holiday..."
...,...,...,...,...,...,...
4079,pov,Ariana Grande,"[2020-11-14, 2020-11-21, 2020-11-28, 2020-12-0...",20,27,It's like you got superpowers\nTurn my minutes...
4080,punchin'.the.clock,J. Cole,"[2021-05-29, 2021-06-05]",2,20,It ain't nothin' I want more\nAin't nothin' I ...
4081,thanK you aIMee,Taylor Swift,"[2024-05-04, 2024-05-11, 2024-05-18]",3,23,"When I picture my hometown\nThere's a bronze, ..."
4082,the.climb.back,J. Cole,"[2021-05-29, 2021-06-05]",2,25,Are you doin' this work to facilitate growth o...


In [3]:
import numpy as np

from src.embeddings.download import download
from src.embeddings.embedder import Embedder
from src.embeddings.embed_texts import embed_texts

from src.ingest_lyrics import load_lyrics
from src.chunk_lyrics import build_chunk_documents
from src.config import MODEL_NAME, MODEL_PATH

download(MODEL_NAME)
embedder = Embedder(MODEL_PATH)

lyrics_df = load_lyrics()
documents = build_chunk_documents(lyrics_df)

texts = [doc["section"] for doc in documents]
vectors = embed_texts(texts, embedder)
X = np.array(vectors)

  exists models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  exists models/Xenova/all-MiniLM-L6-v2/model.onnx
0 problems occured when chunking lyrics.
41681 lyrics chunks generated.


  0%|          | 0/834 [00:00<?, ?it/s]

In [6]:
documents[0]

{'df_song_id': 0,
 'title': '$ex Appeal',
 'performer': 'Baby Keem Featuring Too $hort',
 'section_id': 1,
 'section': "(Play with me, play with me)\n(Play with me, pla-play with me) up all night, baby\n(Play with me, play with me)\n(Play with me, pla-play with me) it's $hort Dog",
 'num_lines': 4}

In [8]:
from src.embeddings.embed_texts import save_embeddings
save_embeddings(vectors)

embeddings shape: (41681, 384)
Saved embeddings to /Users/chs/Documents/LLM-zoomcamp/Capstone/LyricLens/data/processed/embeddings_2020-01-01_2026-08-01.npy.


In [10]:
from src.config import EMBEDDINGS_OUTPUT
vectors = np.load(EMBEDDINGS_OUTPUT)

In [12]:
vectors.shape

(41681, 384)

In [5]:
query = "Grief after losing best friend"
v_query = embedder.encode(query)

scores = X.dot(v_query)
top5 = np.argsort(scores)[-5:][::-1]

for i in top5:
    print(scores[i])
    print(documents[i])

0.5263760866097053
{'df_song_id': 2543, 'title': 'Old Phone', 'performer': 'Ed Sheeran', 'section_id': 7, 'section': "Conversations with my dead friends\nMessages from all my exes\nI kinda think that this was best left\nThere in the past where it belongs\nI feel an overwhelming sadness\nOf all the friends I do not have left\nSeeing how my family has fractured\nGrowin' up and movin' on", 'num_lines': 8}
0.5258248721371901
{'df_song_id': 2543, 'title': 'Old Phone', 'performer': 'Ed Sheeran', 'section_id': 2, 'section': "Conversations with my dead friends\nMessages from all my exes\nI kinda think that this was best left\nIn the past where it belongs\nI feel an overwhelming sadness\nOf all the friends I do not have left\nSeeing how my family has fractured\nGrowin' up and movin' on", 'num_lines': 8}
0.522244341255196
{'df_song_id': 2543, 'title': 'Old Phone', 'performer': 'Ed Sheeran', 'section_id': 4, 'section': "Conversations with my dead friends\nMessages from all my exes\nI kinda think 

---

## 3. PostgreSQL Vector Search

In [15]:
query = "Feeling excited about a new relationship"
query_vector = embedder.encode(query)
query_vector

array([-3.70246856e-02, -9.07255560e-02,  5.18818848e-02,  4.97493561e-03,
        2.33333492e-02, -2.43765977e-02,  5.12082040e-02, -4.34775216e-02,
        1.28409819e-01, -5.00872030e-02, -8.03554480e-03, -2.64668649e-02,
       -4.82610048e-02,  3.44469745e-03,  8.70352327e-02,  7.78248620e-03,
        8.74774661e-03, -6.28760679e-02, -9.49041630e-02,  1.39147379e-03,
       -5.59587385e-02, -1.14810902e-01, -5.34540332e-02, -2.61786172e-02,
       -1.36993443e-03, -9.15241035e-03,  4.00541064e-03, -3.46307730e-03,
       -1.20615155e-02,  1.72795063e-02,  5.68994372e-02, -8.32932533e-03,
        6.57576496e-02, -3.04039189e-02, -4.39538810e-03,  1.16369494e-02,
       -4.43099992e-02, -2.53183926e-04, -2.01048580e-03,  1.68139179e-03,
       -2.51430128e-02,  1.18327313e-02,  4.13956641e-02,  1.26541881e-01,
       -7.61216282e-03,  5.10736790e-03,  7.55031438e-02, -1.41624949e-02,
        1.05707386e-02,  3.12366917e-02, -1.39571591e-02, -1.14708490e-02,
       -4.41172979e-02, -

In [16]:
from src.ingest_lyrics import vec_to_str
query_str = vec_to_str(query_vector)
query_str

'[-0.037024685648331224,-0.09072555595374834,0.0518818847633837,0.004974935611992406,0.02333334922228463,-0.024376597726655416,0.05120820400131439,-0.043477521642071915,0.12840981891235553,-0.05008720304626126,-0.008035544804048285,-0.026466864899636358,-0.048261004795347416,0.0034446974508162208,0.08703523273209393,0.007782486198748433,0.008747746609128805,-0.06287606786576974,-0.09490416295062121,0.0013914737892967816,-0.055958738518460074,-0.11481090214249705,-0.05345403322919783,-0.02617861717333538,-0.0013699344338740922,-0.009152410354739082,0.004005410638663193,-0.0034630772997122296,-0.01206151549390871,0.01727950632266748,0.0568994372464273,-0.008329325334131362,0.06575764956498176,-0.030403918875193447,-0.0043953880966043435,0.011636949419162027,-0.04430999917541478,-0.0002531839258578053,-0.002010485797594301,0.0016813917899221053,-0.025143012790977484,0.01183273129012187,0.041395664133246836,0.12654188104768463,-0.007612162823590213,0.005107367902972751,0.07550314382264614,

In [19]:
from src.ingest_lyrics import setup_postgres, check_tables
conn = setup_postgres()
check_tables(conn)

Connecting to PostgreSQL...
Connected!
pgvector extension ready!
4084 songs and 41681 lyrics chunk documents in database.


(4084, 41681)

In [21]:
from src.config import TOP_K

results = conn.execute(
    """
    SELECT
        d.song_id,
        s.title,
        s.performer,
        s.wks_on_chart,
        s.peak_pos,
        d.section_id,
        d.section,
        d.num_lines,
        1 - (d.embedding <=> %s::vector) AS similarity
    FROM documents AS d
    JOIN songs AS s
        ON d.song_id = s.song_id
    ORDER BY d.embedding <=> %s::vector
    LIMIT %s
    """,
    (query_str, query_str, TOP_K)
).fetchall()

results

[(1141,
  "Fallin'",
  "Why Don't We",
  1,
  37,
  10,
  "Oh, baby, I can feel the rush of adrenaline\nI'm not scared to jump if you want to\nLet's just fall in love for the hell of it\nMaybe, we'll just keep fallin'",
  4,
  0.5218382037233773),
 (1141,
  "Fallin'",
  "Why Don't We",
  1,
  37,
  11,
  "I can feel the rush of adrenaline\nI'm not scared to jump, 'cause I want you\nLet's just fall in love for the hell of it\nMaybe, we'll just keep fallin'",
  4,
  0.5107233232690884),
 (1226,
  'For Tonight',
  'Giveon',
  20,
  61,
  6,
  "We've become numb to what we know is wrong\nBut no one knows but us\nThe feelings rush every single time we touch\nEven though it's what we want, can't keep this up for long",
  4,
  0.48806115913755654),
 (1141,
  "Fallin'",
  "Why Don't We",
  1,
  37,
  6,
  "Oh, baby, I can feel the rush of adrenaline\nI'm not scared to jump if you want to\nLet's just fall in love for the hell of it\nMaybe, we'll just keep fallin'\nI can feel the rush of adrenal

In [22]:
[{"song_id": r[0], "title": r[1], "performer": r[2], "wks_on_chart": r[3], "peak_pos": r[4], "section_id": r[5], "section": r[6], "num_lines": r[7]} for r in results]

[{'song_id': 1141,
  'title': "Fallin'",
  'performer': "Why Don't We",
  'wks_on_chart': 1,
  'peak_pos': 37,
  'section_id': 10,
  'section': "Oh, baby, I can feel the rush of adrenaline\nI'm not scared to jump if you want to\nLet's just fall in love for the hell of it\nMaybe, we'll just keep fallin'",
  'num_lines': 4},
 {'song_id': 1141,
  'title': "Fallin'",
  'performer': "Why Don't We",
  'wks_on_chart': 1,
  'peak_pos': 37,
  'section_id': 11,
  'section': "I can feel the rush of adrenaline\nI'm not scared to jump, 'cause I want you\nLet's just fall in love for the hell of it\nMaybe, we'll just keep fallin'",
  'num_lines': 4},
 {'song_id': 1226,
  'title': 'For Tonight',
  'performer': 'Giveon',
  'wks_on_chart': 20,
  'peak_pos': 61,
  'section_id': 6,
  'section': "We've become numb to what we know is wrong\nBut no one knows but us\nThe feelings rush every single time we touch\nEven though it's what we want, can't keep this up for long",
  'num_lines': 4},
 {'song_id': 1141,

In [37]:
from src.search_pgvector import search_sections
section_results = search_sections(query, embedder, conn)
section_results

[{'document_id': 11683,
  'song_id': 1141,
  'title': "Fallin'",
  'performer': "Why Don't We",
  'wks_on_chart': 1,
  'peak_pos': 37,
  'section_id': 10,
  'section': "Oh, baby, I can feel the rush of adrenaline\nI'm not scared to jump if you want to\nLet's just fall in love for the hell of it\nMaybe, we'll just keep fallin'",
  'num_lines': 4,
  'similarity': 0.5218382037233773},
 {'document_id': 11684,
  'song_id': 1141,
  'title': "Fallin'",
  'performer': "Why Don't We",
  'wks_on_chart': 1,
  'peak_pos': 37,
  'section_id': 11,
  'section': "I can feel the rush of adrenaline\nI'm not scared to jump, 'cause I want you\nLet's just fall in love for the hell of it\nMaybe, we'll just keep fallin'",
  'num_lines': 4,
  'similarity': 0.5107233232690884},
 {'document_id': 12559,
  'song_id': 1226,
  'title': 'For Tonight',
  'performer': 'Giveon',
  'wks_on_chart': 20,
  'peak_pos': 61,
  'section_id': 6,
  'section': "We've become numb to what we know is wrong\nBut no one knows but us\n

In [31]:
[lyrics_df["peak_pos"].min(), lyrics_df["peak_pos"].max()]

[np.int64(1), np.int64(100)]

In [38]:
len(section_results)

20

In [39]:
from src.search_pgvector import aggregate_song_results
top_k_songs = aggregate_song_results(section_results)
top_k_songs

[{'song_id': 1141,
  'title': "Fallin'",
  'performer': "Why Don't We",
  'wks_on_chart': 1,
  'peak_pos': 37,
  'best_similarity': 0.5218382037233773,
  'num_matches': 5,
  'matched_sections': [{'section_id': 10,
    'section': "Oh, baby, I can feel the rush of adrenaline\nI'm not scared to jump if you want to\nLet's just fall in love for the hell of it\nMaybe, we'll just keep fallin'",
    'num_lines': 4,
    'similarity': 0.5218382037233773},
   {'section_id': 11,
    'section': "I can feel the rush of adrenaline\nI'm not scared to jump, 'cause I want you\nLet's just fall in love for the hell of it\nMaybe, we'll just keep fallin'",
    'num_lines': 4,
    'similarity': 0.5107233232690884},
   {'section_id': 6,
    'section': "Oh, baby, I can feel the rush of adrenaline\nI'm not scared to jump if you want to\nLet's just fall in love for the hell of it\nMaybe, we'll just keep fallin'\nI can feel the rush of adrenaline\nI'm not scared to jump, 'cause I want you\nLet's just fall in love

In [40]:
len(top_k_songs)

5